In [ ]:
import os
import subprocess
import re
import csv
from concurrent.futures import ThreadPoolExecutor, as_completed
from openbabel import openbabel
from openpyxl import Workbook
from openbabel import pybel
import shutil
import time

In [ ]:
os.environ['PATH'] = '/home/fwtop/apps/openmpi/bin:' + os.environ.get('PATH', '')

os.environ['LD_LIBRARY_PATH'] = '/home/fwtop/apps/openmpi/lib:' + os.environ.get('LD_LIBRARY_PATH', '')

os.environ['OMPI_ALLOW_RUN_AS_ROOT'] = '1'
os.environ['OMPI_ALLOW_RUN_AS_ROOT_CONFIRM'] = '1'

In [ ]:

MAX_TASKS = 2

In [ ]:

nproc = 10
mem = 3000  

In [ ]:

os.makedirs('dimer_gas/energy/success', exist_ok=True)
os.makedirs('dimer_gas/energy/failure', exist_ok=True)

In [ ]:

opt_freq_path = "dimer_gas/opt+freq/success"
energy_path = "dimer_gas/energy"

In [ ]:

energy_success_dir = os.path.join(energy_path, "success")
energy_failure_dir = os.path.join(energy_path, "failure")
os.makedirs(energy_success_dir, exist_ok=True)
os.makedirs(energy_failure_dir, exist_ok=True)

In [ ]:


def extract_opt_coordinates(xyz_file):
    with open(xyz_file, 'r') as file:
        lines = file.readlines()
    
    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*$')
    
    
    start_index = None
    for index, line in enumerate(lines):
        if atom_line_pattern.match(line):
            start_index = index
            break
    
    
    final_index = None
    for index, line in enumerate(lines[start_index + 1 : ]):
        if atom_line_pattern.match(line) == False:
            final_index = index
            break
    
    
    coordinates_lines = lines[start_index : final_index]
    
    
    coordinates_str = ''.join(coordinates_lines)
    
    return coordinates_str

In [ ]:

def extract_inp_charge_and_multiplicity_line(inp_file):
    with open(inp_file, 'r') as file:
        lines = file.readlines()
    
    
    atom_line_pattern = re.compile(r'^\s*([A-Za-z]{1,2})\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s+([-]?\d+\.\d+)\s*$')
    
    
    start_index = None
    for index, line in enumerate(lines):
        if atom_line_pattern.match(line):
            start_index = index
            break
    
    
    charge_and_multiplicity_line = lines[start_index - 1]        
    
    return charge_and_multiplicity_line

In [ ]:

def create_energy_inp(opt_xyz_file, opt_inp_file, energy_inp_file, mem=mem, nproc=nproc):
    coordinates = extract_opt_coordinates(opt_xyz_file) 
    charge_and_multiplicity_line = extract_inp_charge_and_multiplicity_line(opt_inp_file) 
    
    with open(energy_inp_file, 'w') as f:
        f.write(f"! wB97M-V ma-def2-TZVP autoaux RIJCOSX strongSCF noautostart miniprint\n")
        f.write(f"%maxcore     {mem}\n")
        f.write(f"%pal nprocs  {nproc} end\n")
        f.write(f"%elprop Polar true end\n") 
        f.write(charge_and_multiplicity_line)
        f.write(coordinates)
        f.write(f"*")

In [ ]:
def run_orca_gas_energy(energy_inp_file, energy_success_dir, energy_failure_dir):
    
    
    inp_dir = os.path.dirname(energy_inp_file) if os.path.dirname(energy_inp_file) else '.'
    
    
    base_name = os.path.splitext(os.path.basename(energy_inp_file))[0]
    
    
    output_file = os.path.join(inp_dir, base_name + '.out')
    opt_file = os.path.join(inp_dir, base_name + '.opt')
    trj_file = os.path.join(inp_dir, base_name + '_trj' + '.xyz')
    xyz_file = os.path.join(inp_dir, base_name + '.xyz')
    property_file = os.path.join(inp_dir, base_name + '.property' + '.txt')
    gbw_file = os.path.join(inp_dir, base_name + '.gbw')
    engrad_file = os.path.join(inp_dir, base_name + '.engrad')
    densitiesinfo_file = os.path.join(inp_dir, base_name + '.densitiesinfo')
    densities_file = os.path.join(inp_dir, base_name + '.densities')
    hess_file = os.path.join(inp_dir, base_name + '.hess')
    bibtex_file = os.path.join(inp_dir, base_name + '.bibtex')

    try:
        
        with open(output_file, 'w') as out:
            subprocess.run(['/home/public/orca_6_0_1_linux_x86-64_shared_openmpi416_avx2/orca', energy_inp_file], stdout=out, check=True)

        
        with open(output_file, 'r') as f:
            content = f.read()
            if 'ORCA TERMINATED NORMALLY' in content:
                
                files_to_move = [
                    energy_inp_file, output_file, opt_file, trj_file, xyz_file,
                    property_file, gbw_file, engrad_file, densitiesinfo_file,
                    densities_file, hess_file, bibtex_file
                ]
                for f in files_to_move:
                    if os.path.exists(f):
                        os.rename(f, os.path.join(energy_success_dir, os.path.basename(f)))
                
                new_output_path = os.path.join(energy_success_dir, os.path.basename(output_file))
                return new_output_path, True
            else:
                
                files_to_move = [
                    energy_inp_file, output_file, opt_file, trj_file, xyz_file,
                    property_file, gbw_file, engrad_file, densitiesinfo_file,
                    densities_file, hess_file, bibtex_file
                ]
                for f in files_to_move:
                    if os.path.exists(f):
                        os.rename(f, os.path.join(energy_failure_dir, os.path.basename(f)))
                return None, False

    except subprocess.CalledProcessError as e:
        
        if os.path.exists(energy_inp_file):
            os.rename(energy_inp_file, os.path.join(energy_failure_dir, os.path.basename(energy_inp_file)))
        if os.path.exists(output_file):
            os.rename(output_file, os.path.join(energy_failure_dir, os.path.basename(output_file)))
        print(f"Error running ORCA for file {energy_inp_file}: {str(e)}")
        print(traceback.format_exc())
        return None, False

In [ ]:

def main():
    
    start_time = time.time()
    
    
    for out_file in os.listdir(opt_freq_path):
        if out_file.endswith(".out"):
            
            base_name = os.path.splitext(out_file)[0]

            
            opt_xyz_file = os.path.join(opt_freq_path, base_name + ".xyz")
            opt_inp_file = os.path.join(opt_freq_path, base_name + ".inp")

            
            energy_inp_file = os.path.join(energy_path, out_file.replace(".out", ".inp"))

            
            create_energy_inp(opt_xyz_file, opt_inp_file, energy_inp_file)

    
    energy_inp_files = [
        os.path.join(energy_path, f) for f in os.listdir(energy_path) if f.endswith(".inp")
    ]
    
    
    with ThreadPoolExecutor(max_workers=MAX_TASKS) as executor:
        
        futures = [
            executor.submit(run_orca_gas_energy, inp_file, energy_success_dir, energy_failure_dir)
            for inp_file in energy_inp_files
        ]
        
        for future in as_completed(futures):
            pass  
            
    
    end_time = time.time()
    
    
    elapsed_time = end_time - start_time
    print(f"The code ran for {elapsed_time} seconds.")


In [ ]:
if __name__ == "__main__":
    main()